# TP2 — Modèles génératifs avec PyTorch (MNIST)
**Module :** IA Générative Multimodale et Agentique  
**Objectif :** construire, entraîner, évaluer et explorer un classifieur discriminatif, un auto-encodeur et un VAE sur MNIST.  
**Auteur :** Salma Riad

Ce notebook reprend la **première partie** décrite dans Google Classroom et les architectures du notebook fourni par le professeur. Les expériences sont à exécuter localement : aucun résultat d’entraînement n’est inventé. Les données se téléchargent au premier lancement.

## 0. Préparation (Python 3.11 recommandé par le professeur)
Installer les dépendances dans un environnement Python avec `uv` ou `pip`. Puis sélectionner le noyau de cet environnement dans VS Code.

```powershell
uv init --python 3.11
uv venv
uv add ipykernel torch torchvision matplotlib numpy
```

Sous Windows, on peut exécuter le notebook sans activer manuellement l’environnement, en sélectionnant l’interpréteur `.venv` dans VS Code.

In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 128
EPOCHS_CLASSIFIER = 3
EPOCHS_AE = 5
EPOCHS_VAE = 10  # mettre 20 pour retrouver l'expérience longue du cours
print('PyTorch :', torch.__version__, '| Device :', device)

## 1. Charger MNIST
MNIST : 60 000 images d’entraînement et 10 000 images de test, en niveaux de gris, 28×28. Utiliser `ToTensor()` pour garder les pixels entre 0 et 1.

In [ ]:
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=str(DATA_DIR), train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print('Train :', len(train_dataset), '| Test :', len(test_dataset))
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i, 0], cmap='gray')
    ax.set_title(str(labels[i].item()))
    ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Modèle discriminatif — classifier les chiffres
Le classifieur apprend **P(y|x)** : la classe du chiffre selon l’image. Entrée 784, couche cachée 128 ReLU, sortie 10 logits ; perte CrossEntropyLoss.

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
    def forward(self, x):
        return self.net(x)

classifier = Classifier().to(device)
classifier_loss = nn.CrossEntropyLoss()
classifier_optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
classifier_history = []
for epoch in range(EPOCHS_CLASSIFIER):
    classifier.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        classifier_optimizer.zero_grad()
        loss = classifier_loss(classifier(x), y)
        loss.backward()
        classifier_optimizer.step()
        total_loss += loss.item() * x.size(0)
    mean_loss = total_loss / len(train_dataset)
    classifier_history.append(mean_loss)
    print(f'Classifieur {epoch+1}/{EPOCHS_CLASSIFIER} | loss={mean_loss:.4f}')

In [ ]:
classifier.eval()
correct, count = 0, 0
confusion = torch.zeros(10, 10, dtype=torch.long)
with torch.no_grad():
    for x, y in test_loader:
        pred = classifier(x.to(device)).argmax(dim=1).cpu()
        correct += (pred == y).sum().item()
        count += y.numel()
        for a, b in zip(y, pred):
            confusion[a.long(), b.long()] += 1
accuracy = 100 * correct / count
print(f'Accuracy test : {accuracy:.2f}% ({correct}/{count})')
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(confusion.numpy(), cmap='Blues')
ax.set(xlabel='Prédit', ylabel='Vrai', title='Matrice de confusion — Classifieur MNIST')
fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()
plt.plot(classifier_history, marker='o'); plt.title('Loss du classifieur'); plt.xlabel('Époque'); plt.ylabel('Cross entropy'); plt.grid(alpha=.3); plt.show()
torch.save(classifier.state_dict(), OUTPUT_DIR / 'classifier_mnist.pth')

**Interprétation :** le modèle discriminatif classe des images existantes, mais ne génère pas de nouveaux chiffres. L’accuracy est calculée sur le jeu de test (sans réentraîner sur les données de test).

## 3. Auto-encodeur (AE) — reconstruction et espace latent
L’encodeur transforme l’image en un vecteur latent à **2 dimensions**. Le décodeur reconstruit une image 28×28. Perte MSE ; le modèle n’a pas de contrainte de distribution sur l’espace latent.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784,128), nn.ReLU(), nn.Linear(128,32), nn.ReLU(), nn.Linear(32,latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim,32), nn.ReLU(), nn.Linear(32,128), nn.ReLU(), nn.Linear(128,784), nn.Sigmoid())
    def encode(self, x):
        return self.encoder(x)
    def decode(self, z):
        return self.decoder(z).view(-1,1,28,28)
    def forward(self, x):
        return self.decode(self.encode(x))

ae = Autoencoder(latent_dim=2).to(device)
ae_optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)
ae_history = []
for epoch in range(EPOCHS_AE):
    ae.train()
    loss_sum = 0
    for x, _ in train_loader:
        x = x.to(device)
        ae_optimizer.zero_grad()
        loss = F.mse_loss(ae(x), x)
        loss.backward()
        ae_optimizer.step()
        loss_sum += loss.item() * x.size(0)
    ae_history.append(loss_sum / len(train_dataset))
    print(f'AE {epoch+1}/{EPOCHS_AE} | mse={ae_history[-1]:.5f}')

In [ ]:
ae.eval()
with torch.no_grad():
    x, _ = next(iter(test_loader))
    x = x[:10].to(device)
    recon = ae(x).cpu()
fig, axes = plt.subplots(2, 10, figsize=(15, 3.6))
for i in range(10):
    axes[0,i].imshow(x[i,0].cpu(), cmap='gray'); axes[0,i].axis('off')
    axes[1,i].imshow(recon[i,0], cmap='gray'); axes[1,i].axis('off')
axes[0,0].set_ylabel('Original'); axes[1,0].set_ylabel('AE')
plt.suptitle('Autoencodeur — originaux et reconstructions'); plt.tight_layout(); plt.show()
torch.save(ae.state_dict(), OUTPUT_DIR / 'autoencoder_mnist.pth')

In [ ]:
# Explorer l’espace latent 2D de l’AE sur une partie du jeu de test
ae.eval(); latents, digits = [], []
with torch.no_grad():
    for i, (x, y) in enumerate(test_loader):
        if i >= 20: break
        latents.append(ae.encode(x.to(device)).cpu())
        digits.append(y)
z = torch.cat(latents).numpy(); y = torch.cat(digits).numpy()
plt.figure(figsize=(7, 5)); plt.scatter(z[:,0], z[:,1], c=y, cmap='tab10', s=4, alpha=.6)
plt.colorbar(ticks=range(10)); plt.title('AE : espace latent 2D'); plt.xlabel('z1'); plt.ylabel('z2'); plt.show()

## 4. Variational Autoencoder (VAE)
Au lieu d’un point latent déterministe, le VAE apprend `mu` et `logvar` d’une distribution gaussienne. **Reparameterization trick** : `z = mu + exp(0.5 * logvar) * eps`. La loss combine **reconstruction BCE** et **divergence KL**.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784,256), nn.ReLU(), nn.Linear(256,64), nn.ReLU())
        self.fc_mu = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim,64), nn.ReLU(), nn.Linear(64,256), nn.ReLU(), nn.Linear(256,784), nn.Sigmoid())
    def encode(self,x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    def reparameterize(self, mu, logvar):
        return mu + torch.exp(.5 * logvar) * torch.randn_like(mu)
    def decode(self,z):
        return self.decoder(z).view(-1,1,28,28)
    def forward(self,x):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar)), mu, logvar

def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    rec = F.binary_cross_entropy(x_hat, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return rec + beta * kl, rec, kl

vae = VAE(latent_dim=2).to(device)
vae_optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
vae_history = []
for epoch in range(EPOCHS_VAE):
    vae.train(); total, rec_total, kl_total = 0., 0., 0.
    for x, _ in train_loader:
        x = x.to(device)
        vae_optimizer.zero_grad()
        x_hat, mu, logvar = vae(x)
        loss, rec, kl = vae_loss(x_hat, x, mu, logvar)
        loss.backward(); vae_optimizer.step()
        total += loss.item(); rec_total += rec.item(); kl_total += kl.item()
    vae_history.append((total / len(train_dataset), rec_total / len(train_dataset), kl_total / len(train_dataset)))
    print(f'VAE {epoch+1}/{EPOCHS_VAE} | total={vae_history[-1][0]:.2f} rec={vae_history[-1][1]:.2f} KL={vae_history[-1][2]:.2f}')

In [ ]:
vae.eval()
with torch.no_grad():
    x, _ = next(iter(test_loader))
    x = x[:10].to(device)
    recon, _, _ = vae(x)
    recon = recon.cpu()
fig, axes = plt.subplots(2,10,figsize=(15,3.6))
for i in range(10):
    axes[0,i].imshow(x[i,0].cpu(), cmap='gray'); axes[0,i].axis('off')
    axes[1,i].imshow(recon[i,0], cmap='gray'); axes[1,i].axis('off')
plt.suptitle('VAE — originaux et reconstructions'); plt.tight_layout(); plt.show()

In [ ]:
# Génération : échantillonner z sans fournir d'image d'entrée
vae.eval()
with torch.no_grad():
    samples = vae.decode(torch.randn(20,2,device=device)).cpu()
fig,axes=plt.subplots(2,10,figsize=(15,3.5))
for i,ax in enumerate(axes.flat):
    ax.imshow(samples[i,0], cmap='gray'); ax.axis('off')
plt.suptitle('VAE — 20 chiffres générés'); plt.tight_layout(); plt.show()
torch.save(vae.state_dict(), OUTPUT_DIR / 'vae_mnist.pth')

In [ ]:
# Espace latent 2D et grille de génération
vae.eval(); mus=[]; labs=[]
with torch.no_grad():
    for i,(x,y) in enumerate(test_loader):
        if i>=20: break
        mu,_=vae.encode(x.to(device)); mus.append(mu.cpu()); labs.append(y)
z=torch.cat(mus).numpy(); y=torch.cat(labs).numpy()
plt.figure(figsize=(7,5)); plt.scatter(z[:,0],z[:,1],c=y,cmap='tab10',s=4,alpha=.6)
plt.colorbar(ticks=range(10)); plt.title('VAE : espace latent 2D'); plt.show()
with torch.no_grad():
    gx=np.linspace(-3,3,12); gy=np.linspace(3,-3,12)
    zz=torch.tensor([(a,b) for b in gy for a in gx], dtype=torch.float32,device=device)
    grid=vae.decode(zz).cpu().reshape(12,12,28,28)
canvas=np.block([[grid[i,j].numpy() for j in range(12)] for i in range(12)])
plt.figure(figsize=(9,9)); plt.imshow(canvas,cmap='gray'); plt.axis('off');plt.title('VAE — interpolation dans z');plt.show()

## 5. Conclusion et comparaison
| Modèle | Objectif | Loss | Peut générer sans image d’entrée ? |
|---|---|---|---|
| Classifieur | Reconnaître les chiffres | Cross-entropy | Non |
| Auto-encodeur | Reconstruire les chiffres | MSE | Pas de manière fiable par échantillonnage aléatoire |
| VAE | Reconstruire et générer | BCE + KL | Oui, en échantillonnant le latent |

**À commenter après exécution :** accuracy du classifieur, pertes finales AE/VAE, qualité visuelle des reconstructions et générations. Le support joint contient aussi des développements *GAN* et *diffusion*, à traiter si les autres consignes de Classroom les demandent.